In [2]:
import pandas as pd

fraud_df = pd.read_csv("../data/processed/fraud_processed.csv")
credit_df = pd.read_csv("../data/processed/creditcard_processed.csv")

In [3]:
y_fraud = fraud_df["class"]
X_fraud = fraud_df.drop(columns=["class"])

y_credit = credit_df["Class"]
X_credit = credit_df.drop(columns=["Class"])

In [4]:
from sklearn.model_selection import train_test_split

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud,
    y_fraud,
    test_size=0.2,
    stratify=y_fraud,
    random_state=42
)

In [5]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_credit,
    y_credit,
    test_size=0.2,
    stratify=y_credit,
    random_state=42
)

In [6]:
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [7]:
from sklearn.metrics import average_precision_score

In [14]:
X_train_f.dtypes
X_train_f.isnull().sum().sum()
X_train_f.select_dtypes(include="object").columns

Index(['signup_time', 'purchase_time', 'device_id', 'source', 'browser',
       'sex'],
      dtype='object')

In [15]:
X_train_f = X_train_f.drop(columns=["signup_time", "purchase_time"], errors="ignore")
X_test_f = X_test_f.drop(columns=["signup_time", "purchase_time"], errors="ignore")

In [16]:
X_train_f = pd.get_dummies(X_train_f)
X_test_f = pd.get_dummies(X_test_f)

In [23]:
common_cols = X_train_f.columns.intersection(X_test_f.columns)

X_train_f = X_train_f[common_cols]
X_test_f = X_test_f[common_cols]

In [24]:
X_train_f = X_train_f.fillna(0)
X_test_f = X_test_f.fillna(0)

In [25]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"  # important baseline trick
)

log_model.fit(X_train_f, y_train_f)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [26]:
y_pred = log_model.predict(X_test_f)
y_prob = log_model.predict_proba(X_test_f)[:, 1]

In [27]:
print("F1 Score:", f1_score(y_test_f, y_pred))
print("AUC-PR:", average_precision_score(y_test_f, y_prob))
print(confusion_matrix(y_test_f, y_pred))

F1 Score: 0.15363636363636363
AUC-PR: 0.09329548789511966
[[16006 11387]
 [ 1647  1183]]


In [28]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train_f, y_train_f)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [29]:
y_pred_xgb = xgb_model.predict(X_test_f)
y_prob_xgb = xgb_model.predict_proba(X_test_f)[:, 1]

In [30]:
print("F1 Score:", f1_score(y_test_f, y_pred_xgb))
print("AUC-PR:", average_precision_score(y_test_f, y_prob_xgb))
print(confusion_matrix(y_test_f, y_pred_xgb))

F1 Score: 0.0872776099362202
AUC-PR: 0.23503913068604607
[[27374    19]
 [ 2700   130]]


In [31]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [4, 6, 8],
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1]
}

In [32]:
grid = GridSearchCV(
    XGBClassifier(eval_metric="logloss"),
    param_grid,
    scoring="average_precision",
    cv=3,
    verbose=1
)

grid.fit(X_train_f, y_train_f)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


GridSearchCV(cv=3,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False,
                                     eval_metric='logloss', feature_types=None,
                                     feature_weights=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constraint...
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             param_grid={'learning_rate': [0.05, 0.1], 'max_depth': [4, 6, 8],
                         'n_estimators': [100, 200]},
             scoring='average_precision', verbose=1)

In [33]:
best_model = grid.best_estimator_

In [34]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

In [35]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [37]:
auc_scores = []
f1_scores = []

for train_idx, val_idx in kf.split(X_fraud, y_fraud):

    X_train, X_val = X_fraud.iloc[train_idx], X_fraud.iloc[val_idx]
    y_train, y_val = y_fraud.iloc[train_idx], y_fraud.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        eval_metric="logloss"
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)[:, 1]

    auc_scores.append(average_precision_score(y_val, probs))
    f1_scores.append(f1_score(y_val, preds))

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:signup_time: object, purchase_time: object, device_id: object, source: object, browser: object, sex: object

In [38]:
print("AUC-PR Mean:", np.mean(auc_scores))
print("AUC-PR Std:", np.std(auc_scores))

print("F1 Mean:", np.mean(f1_scores))
print("F1 Std:", np.std(f1_scores))

AUC-PR Mean: nan
AUC-PR Std: nan
F1 Mean: nan
F1 Std: nan


c:\Users\Alienware\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Alienware\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Alienware\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Alienware\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Alienware\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret =